# Cyber Threat Intelligence Agent

An agentic system that monitors CVEs, maps them to MITRE ATT&CK techniques,
matches vulnerabilities against your asset inventory, and produces a prioritized
threat briefing — all driven by OpenAI function calling.

## Why an Agent?

Threat intelligence analysis requires **chaining multiple lookups** — querying the
NVD for new CVEs, cross-referencing MITRE ATT&CK for adversary techniques, checking
which of *your* assets are affected, and finally ranking everything by risk. A
traditional script hard-codes the order; an **agent** lets the LLM decide which
tool to call next based on intermediate results, adapting dynamically when a CVE
turns out to be irrelevant or when an asset match reveals unexpected exposure.

In [ ]:
%pip install openai requests pandas --quiet

In [ ]:
import json, time, textwrap
import requests
import pandas as pd
from openai import OpenAI

client = OpenAI()  # reads OPENAI_API_KEY from env
MODEL = "gpt-4o-mini"

## Sample Asset Inventory

A minimal representation of the organisation's technology surface. Each entry
lists the vendor, product, and version string that will be matched against CPE
data returned by the NVD.

In [ ]:
ASSET_INVENTORY = {
    "web-server-01": {
        "vendor": "apache",
        "product": "http_server",
        "version": "2.4.51",
        "criticality": "high",
        "environment": "production",
    },
    "app-server-01": {
        "vendor": "apache",
        "product": "log4j",
        "version": "2.14.1",
        "criticality": "critical",
        "environment": "production",
    },
    "db-server-01": {
        "vendor": "postgresql",
        "product": "postgresql",
        "version": "13.4",
        "criticality": "critical",
        "environment": "production",
    },
    "dev-laptop-42": {
        "vendor": "microsoft",
        "product": "windows_10",
        "version": "21H2",
        "criticality": "medium",
        "environment": "corporate",
    },
    "vpn-gateway-01": {
        "vendor": "openvpn",
        "product": "openvpn",
        "version": "2.5.3",
        "criticality": "high",
        "environment": "perimeter",
    },
}

pd.DataFrame.from_dict(ASSET_INVENTORY, orient="index")

## Tool 1 — `fetch_cves`

Queries the NIST National Vulnerability Database (NVD) REST API v2.0 for CVEs
matching a keyword and optional severity filter.

In [ ]:
def fetch_cves(keyword: str, severity: str = "HIGH") -> str:
    """Search the NVD for CVEs matching *keyword* with at least *severity*."""
    url = "https://services.nvd.nist.gov/rest/json/cves/2.0"
    params = {
        "keywordSearch": keyword,
        "cvssV3Severity": severity.upper(),
        "resultsPerPage": 5,
    }
    try:
        resp = requests.get(url, params=params, timeout=30)
        resp.raise_for_status()
        data = resp.json()
    except Exception as exc:
        return json.dumps({"error": str(exc)})

    results = []
    for vuln in data.get("vulnerabilities", []):
        cve = vuln.get("cve", {})
        cve_id = cve.get("id", "unknown")
        descriptions = cve.get("descriptions", [])
        desc = next((d["value"] for d in descriptions if d["lang"] == "en"), "")
        metrics = cve.get("metrics", {})
        score = None
        for key in ("cvssMetricV31", "cvssMetricV30"):
            if key in metrics:
                score = metrics[key][0]["cvssData"]["baseScore"]
                break
        results.append({"cve_id": cve_id, "description": desc[:300], "cvss_score": score})

    return json.dumps(results, indent=2)

## Tool 2 — `query_mitre_attack`

Fetches a MITRE ATT&CK technique description from the public STIX 2.1 bundle
hosted on GitHub.

In [ ]:
# Cache the MITRE ATT&CK STIX bundle in memory so we only download once
_MITRE_CACHE = None

def _load_mitre_bundle():
    global _MITRE_CACHE
    if _MITRE_CACHE is None:
        url = (
            "https://raw.githubusercontent.com/mitre/cti/master/"
            "enterprise-attack/enterprise-attack.json"
        )
        resp = requests.get(url, timeout=60)
        resp.raise_for_status()
        _MITRE_CACHE = resp.json()
    return _MITRE_CACHE


def query_mitre_attack(technique_id: str) -> str:
    """Return name, description, and tactics for a MITRE ATT&CK technique ID (e.g. T1190)."""
    try:
        bundle = _load_mitre_bundle()
    except Exception as exc:
        return json.dumps({"error": str(exc)})

    for obj in bundle.get("objects", []):
        if obj.get("type") != "attack-pattern":
            continue
        refs = obj.get("external_references", [])
        for ref in refs:
            if ref.get("external_id") == technique_id:
                tactics = [
                    p.get("phase_name", "")
                    for p in obj.get("kill_chain_phases", [])
                ]
                return json.dumps(
                    {
                        "technique_id": technique_id,
                        "name": obj.get("name"),
                        "description": (obj.get("description", ""))[:500],
                        "tactics": tactics,
                    },
                    indent=2,
                )
    return json.dumps({"error": f"Technique {technique_id} not found"})

## Tool 3 — `check_asset_match`

Given a CVE ID, fetch its CPE match criteria from the NVD and compare against
the local asset inventory to find affected hosts.

In [ ]:
def check_asset_match(cve_id: str, assets: dict | None = None) -> str:
    """Check which assets in the inventory are potentially affected by *cve_id*."""
    assets = assets or ASSET_INVENTORY
    url = "https://services.nvd.nist.gov/rest/json/cves/2.0"
    params = {"cveId": cve_id}
    try:
        resp = requests.get(url, params=params, timeout=30)
        resp.raise_for_status()
        data = resp.json()
    except Exception as exc:
        return json.dumps({"error": str(exc)})

    # Collect vendor/product tokens from CPE URIs in the response
    cpe_tokens = set()
    for vuln in data.get("vulnerabilities", []):
        configs = vuln.get("cve", {}).get("configurations", [])
        for cfg in configs:
            for node in cfg.get("nodes", []):
                for match in node.get("cpeMatch", []):
                    uri = match.get("criteria", "")
                    parts = uri.split(":")
                    if len(parts) >= 5:
                        cpe_tokens.add(parts[3].lower())  # vendor
                        cpe_tokens.add(parts[4].lower())  # product

    matched = []
    for host, info in assets.items():
        v = info["vendor"].lower()
        p = info["product"].lower()
        if v in cpe_tokens or p in cpe_tokens:
            matched.append({"host": host, **info, "cve_id": cve_id})

    return json.dumps(
        {"cve_id": cve_id, "cpe_tokens_found": sorted(cpe_tokens), "affected_assets": matched},
        indent=2,
    )

## Tool 4 — `prioritize_threats`

A local scoring function that combines CVSS score, asset criticality, and
environment exposure into a single priority ranking.

In [ ]:
def prioritize_threats(findings: list[dict]) -> str:
    """Score and rank a list of findings. Each dict should contain
    cve_id, cvss_score, asset_criticality ('low'|'medium'|'high'|'critical'),
    and environment ('production'|'corporate'|'perimeter'|'development').
    """
    crit_weight = {"low": 1, "medium": 2, "high": 3, "critical": 4}
    env_weight = {"development": 1, "corporate": 2, "perimeter": 3, "production": 4}

    scored = []
    for f in findings:
        cvss = f.get("cvss_score") or 5.0
        crit = crit_weight.get(f.get("asset_criticality", "medium"), 2)
        env = env_weight.get(f.get("environment", "corporate"), 2)
        priority_score = round(cvss * 0.5 + crit * 1.5 + env * 1.0, 2)
        scored.append({**f, "priority_score": priority_score})

    scored.sort(key=lambda x: x["priority_score"], reverse=True)
    for rank, item in enumerate(scored, 1):
        item["rank"] = rank

    return json.dumps(scored, indent=2)

## Tool Schema for OpenAI Function Calling

Each tool is described in the `tools` list using the format expected by
`chat.completions.create`.

In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "fetch_cves",
            "description": "Search the NVD for CVEs matching a keyword and minimum severity (LOW, MEDIUM, HIGH, CRITICAL).",
            "parameters": {
                "type": "object",
                "properties": {
                    "keyword": {"type": "string", "description": "Search term, e.g. 'apache log4j'"},
                    "severity": {
                        "type": "string",
                        "enum": ["LOW", "MEDIUM", "HIGH", "CRITICAL"],
                        "description": "Minimum CVSS v3 severity",
                    },
                },
                "required": ["keyword"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "query_mitre_attack",
            "description": "Look up a MITRE ATT&CK technique by its ID (e.g. T1190) and return its name, description, and associated tactics.",
            "parameters": {
                "type": "object",
                "properties": {
                    "technique_id": {"type": "string", "description": "ATT&CK technique ID, e.g. T1190"},
                },
                "required": ["technique_id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "check_asset_match",
            "description": "Given a CVE ID, check which assets in our inventory are potentially affected.",
            "parameters": {
                "type": "object",
                "properties": {
                    "cve_id": {"type": "string", "description": "CVE identifier, e.g. CVE-2021-44228"},
                },
                "required": ["cve_id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "prioritize_threats",
            "description": "Score and rank a list of threat findings by combining CVSS score, asset criticality, and environment exposure.",
            "parameters": {
                "type": "object",
                "properties": {
                    "findings": {
                        "type": "array",
                        "items": {
                            "type": "object",
                            "properties": {
                                "cve_id": {"type": "string"},
                                "cvss_score": {"type": "number"},
                                "asset_criticality": {"type": "string"},
                                "environment": {"type": "string"},
                            },
                        },
                        "description": "List of findings to rank",
                    }
                },
                "required": ["findings"],
            },
        },
    }
]

## Dispatcher

Maps function names returned by the model to the actual Python callables.

In [ ]:
TOOL_MAP = {
    "fetch_cves": fetch_cves,
    "query_mitre_attack": query_mitre_attack,
    "check_asset_match": check_asset_match,
    "prioritize_threats": prioritize_threats,
}

## Agent Loop

The core loop sends the conversation to the model, checks for tool calls,
executes them, appends the results, and repeats until the model produces a
final text response.

In [ ]:
SYSTEM_PROMPT = textwrap.dedent("""\
    You are a Cyber Threat Intelligence analyst agent. Your job is to:
    1. Search for recent CVEs relevant to the user's query.
    2. Check which of our assets are affected.
    3. Look up associated MITRE ATT&CK techniques for context.
    4. Prioritize the findings and produce a concise threat briefing.

    Use the provided tools to gather data before producing your final report.
    Always call prioritize_threats before writing the final briefing so that
    findings are ranked by risk.
""")


def run_agent(user_message: str, max_turns: int = 10) -> str:
    """Run the agent loop and return the final assistant message."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_message},
    ]

    for turn in range(max_turns):
        print(f"\n--- Agent turn {turn + 1} ---")
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools,
            tool_choice="auto",
        )
        msg = response.choices[0].message
        messages.append(msg)

        # If there are no tool calls, the model is done
        if not msg.tool_calls:
            print("\n--- Agent finished ---")
            return msg.content

        # Execute each tool call
        for tc in msg.tool_calls:
            fn_name = tc.function.name
            fn_args = json.loads(tc.function.arguments)
            print(f"  Calling {fn_name}({fn_args})")

            fn = TOOL_MAP.get(fn_name)
            if fn is None:
                result = json.dumps({"error": f"Unknown tool: {fn_name}"})
            else:
                result = fn(**fn_args)

            messages.append(
                {"role": "tool", "tool_call_id": tc.id, "content": result}
            )
            # Respect NVD rate limit (6 requests / minute without API key)
            time.sleep(1)

    return "Agent reached maximum turns without finishing."

## Example Execution

We ask the agent to investigate **Apache Log4j** vulnerabilities and assess
the impact on our asset inventory.

In [ ]:
report = run_agent(
    "Investigate critical Apache Log4j vulnerabilities. "
    "Check which of our assets are affected, look up relevant MITRE ATT&CK "
    "techniques (try T1190 and T1059), and give me a prioritized threat briefing."
)

In [ ]:
from IPython.display import Markdown
Markdown(report)

## Second Scenario — OpenVPN Perimeter Scan

Run the agent with a different query to show how it adapts its tool usage.

In [ ]:
report2 = run_agent(
    "Search for HIGH-severity OpenVPN CVEs. Check our perimeter assets "
    "for exposure and look up ATT&CK technique T1133 (External Remote Services). "
    "Prioritize and brief me."
)

In [ ]:
Markdown(report2)

## Results Analysis

### Key Design Observations

| Aspect | Detail |
|--------|--------|
| **Data sources** | NVD REST API v2.0 (live), MITRE ATT&CK STIX bundle (live from GitHub) |
| **Adaptive flow** | The agent chose which CVEs to drill into based on NVD results, rather than following a fixed script |
| **Tool orchestration** | The LLM decided when to call `check_asset_match` vs. `query_mitre_attack`, and it aggregated findings before calling `prioritize_threats` |
| **Limitations** | NVD rate limit (6 req/min unauthenticated); CPE matching is approximate; STIX bundle download is ~30 MB |

### Potential Extensions

1. **Enrich with Exploit-DB** — add a tool that checks whether a public exploit exists.
2. **SBOM integration** — replace the static asset dict with a real Software Bill of Materials.
3. **Slack/Teams alerting** — add a `send_alert` tool so the agent can notify the SOC.
4. **Memory** — persist past briefings so the agent can track *changes* in threat posture.

---
*Notebook by the LLM Course — Agentic Designs series*